In [0]:
from pyspark.sql.functions import lit

In [0]:
%run "../config/snp-cofig"

In [0]:
%run "../config/containers"

In [0]:
dbutils.widgets.text("folder","2021-03-21")

In [0]:
folder=dbutils.widgets.get("folder")

In [0]:
path = raw_container
display(dbutils.fs.ls(path))

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType 

In [0]:
circuits_schema=StructType(
    fields=[
            StructField("circuitId", IntegerType(), False),
            StructField("circuitRef", StringType(), True),
            StructField("name", StringType(), True),
            StructField("location", StringType(), True),
            StructField("country", StringType(), True),
            StructField("lat", DoubleType(), True),
            StructField("lng", DoubleType(), True),
            StructField("alt", IntegerType(), True),
            StructField("url", StringType(), True)
    ]
)

In [0]:
df=spark.read.option("header",True) \
.schema(circuits_schema) \
.csv(f"{raw_container}/{folder}/circuits.csv")
display(df)

In [0]:
df.printSchema()

In [0]:
df.describe().show()

In [0]:
from pyspark.sql.functions import col

In [0]:
selected_df=df.select(
    col("circuitId"),
    col("circuitRef"),
    col("name"),
    col("location"),
    col("country"),
    col("lat"),
    col("lng"),
    col("alt")
)

In [0]:
display(selected_df)

In [0]:
renamed_df=selected_df.withColumnRenamed("circuitId","circuit_id") \
.withColumnRenamed("circuitRef","circuit_ref") \
.withColumnRenamed("lat","latitude") \
.withColumnRenamed("lng","longitude") \
.withColumnRenamed("alt","altitude")

In [0]:
display(renamed_df)

In [0]:
from pyspark.sql.functions import current_timestamp
final_df=renamed_df.withColumn("ingestion_date",current_timestamp()) \
    .withColumn("file_date",lit(folder))
display(final_df)

In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "false") \
    .saveAsTable("api_formula1.default.circuits_table")


In [0]:
dbutils.notebook.exit("1. Data written to SQL warehouse")